In [1]:
!pip install -q gradio chromadb google-generativeai pypdf sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/6

In [2]:
import os
import uuid

import gradio as gr
import chromadb
import google.generativeai as genai
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
os.environ["GEMINI_API_KEY"] = "YOUR_GEMINI_API_KEY"

In [5]:
import os
import uuid

import gradio as gr
import chromadb
import google.generativeai as genai
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

# ==========================
# Gemini API
# ==========================
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
llm = genai.GenerativeModel("gemini-2.5-flash")

# ==========================
# Embedding Model
# ==========================
print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded!")

# ==========================
# ChromaDB
# ==========================
client = chromadb.Client()
collection = client.get_or_create_collection("rag_docs")

# ==========================
# Extract PDF Text
# ==========================
def extract_text(pdf):
    reader = PdfReader(pdf.name)
    text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"

    return text


# ==========================
# Split into Chunks
# ==========================
def chunk_text(text, size=300):
    return [text[i:i + size] for i in range(0, len(text), size)]


# ==========================
# Upload & Index PDF
# ==========================
def upload_pdf(pdf):
    global collection

    if pdf is None:
        return "Please upload a PDF."

    text = extract_text(pdf)

    if text.strip() == "":
        return "No text found in PDF."

    chunks = chunk_text(text)

    print(f"Chunks Created: {len(chunks)}")

    embeddings = embedder.encode(chunks).tolist()

    try:
        client.delete_collection("rag_docs")
    except:
        pass

    collection = client.get_or_create_collection("rag_docs")

    collection.add(
        documents=chunks,
        embeddings=embeddings,
        ids=[str(uuid.uuid4()) for _ in chunks]
    )

    return f"✅ PDF Indexed Successfully ({len(chunks)} chunks)"


# ==========================
# Ask Question
# ==========================
def ask(question):
    global collection

    if question.strip() == "":
        return "Please enter a question."

    query_embedding = embedder.encode(question).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5
    )

    documents = results.get("documents", [])

    if not documents or len(documents[0]) == 0:
        return "I couldn't find the answer in the document."

    context = "\n".join(documents[0])

    context = context[:4000]

    prompt = f"""
You are a helpful AI assistant.

Answer ONLY using the context below.

If the answer is not found, reply exactly:

I couldn't find the answer in the document.

Context:
{context}

Question:
{question}
"""

    response = llm.generate_content(prompt)

    return response.text


# ==========================
# Gradio UI
# ==========================
with gr.Blocks() as demo:

    gr.Markdown("# 📄 RAG PDF Chatbot")

    pdf = gr.File(label="Upload PDF")

    upload_btn = gr.Button("Index PDF")

    status = gr.Textbox(label="Status")

    upload_btn.click(
        fn=upload_pdf,
        inputs=pdf,
        outputs=status
    )

    question = gr.Textbox(label="Ask a Question")

    ask_btn = gr.Button("Ask")

    answer = gr.Textbox(
        label="Answer",
        lines=10
    )

    ask_btn.click(
        fn=ask,
        inputs=question,
        outputs=answer
    )


# ==========================
# Launch
# ==========================
demo.launch(
    server_name="0.0.0.0",
    server_port=int(os.environ.get("PORT", 7860))
)

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4b07f225259b160306.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
